# Hybrid Search with Reranking: Best-Accuracy Concept Mapping

This notebook demonstrates **hybrid search with cross-encoder reranking** -- the highest-accuracy
configuration for OMOP CDM concept mapping using the Portiere SDK. We combine BM25s (lexical
retrieval) and FAISS (semantic retrieval) with Reciprocal Rank Fusion (RRF), then rerank
candidates using a cross-encoder model for fine-grained precision.

## Approach Overview

Standard retrieval backends each have strengths and weaknesses. Lexical search (BM25s) excels
when source codes use exact terminology from the target vocabulary. Semantic search (FAISS with
biomedical embeddings) handles synonyms, abbreviations, and paraphrases. By combining both
and adding a cross-encoder reranker, we get the best of all worlds.

| Technique | Lexical | Semantic | Fusion | Reranking | Accuracy |
|-----------|---------|----------|--------|-----------|----------|
| BM25s only | Yes | No | No | No | Good |
| FAISS only | No | Yes | No | No | Better |
| Hybrid (RRF) | Yes | Yes | RRF | No | Great |
| Hybrid + Reranker | Yes | Yes | RRF | Cross-encoder | Best |

### Reciprocal Rank Fusion (RRF)

RRF merges ranked lists from multiple retrieval backends into a single fused ranking. For each
candidate concept, the fused score is computed as:

```
score = sum(1 / (k + rank_i)) across all backends
```

where `k` is a smoothing constant (default 60) and `rank_i` is the candidate's position in
backend `i`. Concepts ranked highly by both BM25s and FAISS receive the highest fused scores,
while concepts found by only one backend still receive credit.

### Cross-Encoder Reranking

After RRF fusion produces a candidate list, a cross-encoder model (`cross-encoder/ms-marco-MiniLM-L-6-v2`)
re-scores each (query, candidate) pair. Unlike bi-encoders that embed query and candidate
independently, cross-encoders jointly encode both texts through a single transformer pass,
enabling fine-grained token-level comparison.

The final blended score combines both signals:

```
final_score = 0.6 * sigmoid(cross_encoder_logit) + 0.4 * rrf_score
```

This preserves the diversity of retrieval (40% weight) while letting the cross-encoder
make fine-grained corrections (60% weight).

## What We Will Do

1. Install dependencies (FAISS + sentence-transformers)
2. Build indexes from the OHDSI Athena vocabulary (SNOMED, LOINC, RxNorm, ICD10CM)
3. Configure hybrid search with RRF fusion and cross-encoder reranking
4. Create a project targeting OMOP CDM v5.4
5. Add patient and diagnosis data sources
6. Map schema and approve
7. Map diagnosis codes with hybrid search + reranking
8. Inspect RRF and cross-encoder scores
9. Compare with single-backend results
10. Run ETL and validate

## Data Files

All data files are in `example_assets/01.3_hybrid_search_reranking/`:

- `patients.csv` -- 15 patients, 11 demographic columns
- `diagnoses.csv` -- 20 diagnosis records with ICD-10 codes (E11.9, I10, J06.9, J44.1, N18.3, R51, F32.9, Z87.891)

Vocabulary indexes are built from the shared OHDSI Athena vocabulary in `example_assets/vocabulary_download_v5/`.

## Step 1: Install Dependencies

The hybrid + reranker configuration requires FAISS for dense retrieval and
sentence-transformers for both embedding and cross-encoder models.

In [1]:
!pip install portiere[faiss] sentence-transformers

zsh:1: no matches found: portiere[faiss]


## Step 2: Build Indexes from Athena Vocabulary

Before we can search, we need to build FAISS and BM25s indexes from the OHDSI Athena
vocabulary. This encodes each concept name into a dense vector using SapBERT and stores
the vectors in a FAISS index for fast similarity search.

The `hybrid_backends` parameter explicitly specifies which sub-backends to combine.
Here we use `["bm25s", "faiss"]` -- the classic combination. You can also combine
BM25s with ChromaDB, Qdrant, Milvus, PGVector, or MongoDB. See `04_knowledge_backends.ipynb`
for all supported backends.

This step runs once on first execution (~10-30 min for FAISS embedding). On subsequent
runs, the existing indexes are reused from disk.

In [2]:
from pathlib import Path
from portiere.knowledge import build_knowledge_layer

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

# Build indexes from Athena on first run (~10-30 min for FAISS embedding)
if not BM25S_CORPUS.exists():
    paths = build_knowledge_layer(
        athena_path=ATHENA_DIR,
        output_path=VOCAB_DIR,
        backend="hybrid",
        hybrid_backends=["bm25s", "faiss"],
        vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    )
    print(f"Built indexes: {paths}")
else:
    print(f"Using existing indexes at {VOCAB_DIR}")

Using existing indexes at example_assets/vocabulary_download_v5/indexes


## Step 3: Import and Configure

We configure Portiere with:

- `knowledge_layer` with `backend="hybrid"` -- combines BM25s and FAISS retrieval
- `fusion_method="rrf"` with `rrf_k=60` -- Reciprocal Rank Fusion with standard smoothing
- `embedding` with SapBERT -- biomedical sentence embeddings for FAISS
- `reranker` with cross-encoder -- fine-grained candidate reranking

Portiere infers **local mode** automatically when a `knowledge_layer` is configured and no
`api_key` is provided. There is no need to set `mode` or `pipeline` explicitly.

In [3]:
from pathlib import Path

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

In [4]:
from portiere.config import (
    PortiereConfig,
    KnowledgeLayerConfig,
    EmbeddingConfig,
    RerankerConfig,
)

config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="hybrid",
        hybrid_backends=["bm25s", "faiss"],
        bm25s_corpus_path=str(BM25S_CORPUS),
        faiss_index_path=str(FAISS_INDEX),
        faiss_metadata_path=str(FAISS_META),
        fusion_method="rrf",
        rrf_k=60,
    ),
    embedding=EmbeddingConfig(
        provider="huggingface",
        model="cambridgeltl/SapBERT-from-PubMedBERT-fulltext",
    ),
    reranker=RerankerConfig(
        provider="huggingface",
        model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    ),
)

print(f"Backend: {config.knowledge_layer.backend}")
print(f"Sub-backends: {config.knowledge_layer.hybrid_backends}")
print(f"Fusion: {config.knowledge_layer.fusion_method} (k={config.knowledge_layer.rrf_k})")
print(f"Embedding: {config.embedding.provider}/{config.embedding.model}")
print(f"Reranker: {config.reranker.provider}/{config.reranker.model}")

Backend: hybrid
Sub-backends: ['bm25s', 'faiss']
Fusion: rrf (k=60)
Embedding: huggingface/cambridgeltl/SapBERT-from-PubMedBERT-fulltext
Reranker: huggingface/cross-encoder/ms-marco-MiniLM-L-6-v2


## Step 4: Create a Project

Initialize a new mapping project targeting OMOP CDM v5.4. The project uses the hybrid
configuration we defined above for all concept mapping operations.

In [5]:
import portiere
from portiere.engines import PolarsEngine

project = portiere.init(
    name="Hybrid Search Demo",
    engine=PolarsEngine(),
    target_model="omop_cdm_v5.4",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    config=config,
)

print(project)

2026-04-16 00:31:02 [info     ] PolarsEngine initialized      


2026-04-16 00:31:02 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


Project(name='Hybrid Search Demo', task='standardize', target_model='omop_cdm_v5.4', mode='local')


## Step 5: Add Data Sources

We add two CSV files:

- `patients.csv` -- 15 patients with 11 demographic columns (patient_id, name, DOB, gender, race, etc.)
- `diagnoses.csv` -- 20 diagnosis records with ICD-10 codes

The SDK will read each file, detect its format, and profile the columns.

In [6]:
patients_source = project.add_source(
    "example_assets/01.3_hybrid_search_reranking/patients.csv"
)

print(f"Source added: {patients_source['name']}, format: {patients_source['format']}")
print(f"Columns detected: {patients_source.get('columns', 'N/A')}")
print(f"Row count: {patients_source.get('row_count', 'N/A')}")

2026-04-16 00:31:02 [info     ] project.source_added           project='Hybrid Search Demo' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:31:05 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


2026-04-16 00:31:05 [info     ] project.profiled               source=patients


Source added: patients, format: csv
Columns detected: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'address', 'city', 'state', 'zip_code']
Row count: 15


In [7]:
diagnoses_source = project.add_source(
    "example_assets/01.3_hybrid_search_reranking/diagnoses.csv"
)

print(f"Source added: {diagnoses_source['name']}, format: {diagnoses_source['format']}")
print(f"Columns detected: {diagnoses_source.get('columns', 'N/A')}")
print(f"Row count: {diagnoses_source.get('row_count', 'N/A')}")

2026-04-16 00:31:05 [info     ] project.source_added           project='Hybrid Search Demo' source=diagnoses


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:31:05 [info     ] gx_profiler.profiled           columns=6 rows=20 source=diagnoses


2026-04-16 00:31:05 [info     ] project.profiled               source=diagnoses


Source added: diagnoses, format: csv
Columns detected: ['encounter_id', 'patient_id', 'diagnosis_code', 'diagnosis_description', 'diagnosis_type', 'diagnosis_date']
Row count: 20


## Step 6: Map Schema

Schema mapping automatically matches source columns to OMOP CDM target tables and columns.
The SDK uses the knowledge layer to find the best matches and assigns confidence scores.

Confidence routing determines the status of each mapping:

| Confidence Range | Status | Action |
|-----------------|--------|--------|
| >= 0.95 | AUTO_ACCEPTED | No review needed |
| 0.80 - 0.95 | VERIFY | Quick verification recommended |
| 0.70 - 0.80 | REVIEW | Manual review needed |
| < 0.70 | MANUAL | Requires manual mapping |

In [8]:
schema_map = project.map_schema(patients_source)

print(schema_map)
print(f"\nSummary: {schema_map.summary()}")

2026-04-16 00:31:05 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=omop_cdm_v5.4


2026-04-16 00:31:05 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:31:05 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:31:05 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:31:08 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:31:08 [info     ] Stage 2 complete               auto=10 review=0 total=11


2026-04-16 00:31:08 [info     ] local_storage.schema_mapping_saved items_count=11 project='Hybrid Search Demo'


2026-04-16 00:31:08 [info     ] project.schema_mapped          auto=10 project='Hybrid Search Demo' total=11


items=[SchemaMappingItem(source_column='patient_id', source_table='', target_table='person', target_column='person_id', confidence=0.95, status=<MappingStatus.AUTO_ACCEPTED: 'auto_accepted'>, candidates=[], override_target_table=None, override_target_column=None), SchemaMappingItem(source_column='first_name', source_table='', target_table='person', target_column='person_source_value', confidence=0.95, status=<MappingStatus.AUTO_ACCEPTED: 'auto_accepted'>, candidates=[], override_target_table=None, override_target_column=None), SchemaMappingItem(source_column='last_name', source_table='', target_table='person', target_column='person_source_value', confidence=0.5, status=<MappingStatus.UNMAPPED: 'unmapped'>, candidates=[], override_target_table=None, override_target_column=None), SchemaMappingItem(source_column='date_of_birth', source_table='', target_table='person', target_column='birth_datetime', confidence=0.95, status=<MappingStatus.AUTO_ACCEPTED: 'auto_accepted'>, candidates=[], ove

### Inspect Schema Mapping Results

Each item shows the source column, target table/column, confidence score, and mapping status.

In [9]:
print("Schema Mapping Results:")
print("-" * 80)

for item in schema_map.items:
    print(
        f"  {item.source_column} -> {item.target_table}.{item.target_column} "
        f"(confidence: {item.confidence:.2f}, status: {item.status.value})"
    )

Schema Mapping Results:
--------------------------------------------------------------------------------
  patient_id -> person.person_id (confidence: 0.95, status: auto_accepted)
  first_name -> person.person_source_value (confidence: 0.95, status: auto_accepted)
  last_name -> person.person_source_value (confidence: 0.50, status: unmapped)
  date_of_birth -> person.birth_datetime (confidence: 0.95, status: auto_accepted)
  gender -> person.gender_concept_id (confidence: 0.95, status: auto_accepted)
  race -> person.race_concept_id (confidence: 0.95, status: auto_accepted)
  ethnicity -> person.ethnicity_concept_id (confidence: 0.95, status: auto_accepted)
  address -> location.address_1 (confidence: 0.95, status: auto_accepted)
  city -> location.city (confidence: 0.95, status: auto_accepted)
  state -> location.state (confidence: 0.95, status: auto_accepted)
  zip_code -> location.zip (confidence: 0.95, status: auto_accepted)


## Step 7: Review and Approve Schema

After reviewing, approve all mappings and finalize the schema mapping.

- `approve_all()` -- marks all remaining items as approved
- `finalize()` -- locks the schema mapping so it can be used in ETL generation

In [10]:
schema_map.approve_all()
schema_map.finalize()

print("Schema mapping finalized.")
print(f"Updated summary: {schema_map.summary()}")

Schema mapping finalized.
Updated summary: {'total': 11, 'auto_accepted': 10, 'needs_review': 0, 'approved': 0, 'unmapped': 1, 'auto_rate': 90.9090909090909}


## Step 8: Map Concepts with Hybrid Search + Reranking

This is the key step. When we call `map_concepts()` with the hybrid + reranker configuration,
the following pipeline executes for each source code:

1. **BM25s retrieval** -- lexical search over the concept corpus, returns top-k candidates
2. **FAISS retrieval** -- dense vector search using SapBERT embeddings, returns top-k candidates
3. **RRF fusion** -- merges both candidate lists using Reciprocal Rank Fusion
4. **Cross-encoder reranking** -- scores each (source_code, candidate) pair with the cross-encoder
5. **Score blending** -- final score = 0.6 * sigmoid(CE) + 0.4 * rrf_score
6. **Confidence routing** -- assigns mapping status based on the final blended score

We pass the `diagnoses_source` so Portiere auto-discovers columns containing clinical codes
(columns ending in `_code` like `diagnosis_code`).

In [11]:
concept_map = project.map_concepts(source=diagnoses_source)

print(concept_map)
print(f"\nSummary: {concept_map.summary()}")

2026-04-16 00:31:08 [info     ] project.code_columns_detected  columns=['diagnosis_code']


2026-04-16 00:31:08 [info     ] Stage 3: Concept mapping (local) columns=['diagnosis_code'] knowledge_backend=hybrid


2026-04-16 00:31:08 [info     ] Mapping column: diagnosis_code


2026-04-16 00:31:08 [info     ] Found description column: diagnosis_description code_column=diagnosis_code desc_column=diagnosis_description mappings=8


2026-04-16 00:31:08 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


...

2026-04-16 00:31:20 [warning  ] hybrid.backend_failed          backend=LocalFAISSBackend error='sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers'


2026-04-16 00:31:20 [info     ] Stage 3 complete               auto_rate=100.0% total_codes=8


2026-04-16 00:31:20 [info     ] local_storage.concept_mapping_saved items_count=8 project='Hybrid Search Demo'


2026-04-16 00:31:20 [info     ] project.concepts_mapped        auto_rate=100.0 project='Hybrid Search Demo' total=8


items=[ConceptMappingItem(source_code='I10', source_description='Essential (primary) hypertension', source_column='diagnosis_code', source_count=4, target_concept_id=320128, target_concept_name='Essential hypertension', target_vocabulary_id='SNOMED', target_domain_id='Condition', confidence=8.698995590209961, method=<ConceptMappingMethod.AUTO: 'auto'>, candidates=[ConceptCandidate(concept_id=320128, concept_name='Essential hypertension', vocabulary_id='SNOMED', domain_id='Condition', concept_class_id='Disorder', standard_concept='S', score=8.698995590209961, rrf_score=8.698995590209961, cross_encoder_score=None), ConceptCandidate(concept_id=4180283, concept_name='Systolic essential hypertension', vocabulary_id='SNOMED', domain_id='Condition', concept_class_id='Disorder', standard_concept='S', score=7.7399091720581055, rrf_score=7.7399091720581055, cross_encoder_score=None), ConceptCandidate(concept_id=317898, concept_name='Malignant essential hypertension', vocabulary_id='SNOMED', doma

## Step 9: Inspect Mapping Results with Scores

Each concept mapping item includes the source code, the matched target concept, the mapping
method, and the confidence score. The confidence score reflects the blended hybrid + reranker
score after all pipeline stages.

In [12]:
print("Concept Mapping Results:")
print("-" * 80)

for item in concept_map.items:
    print(
        f"Source Code = {item.source_code}\n"
        f"Source Name = {item.source_description}\n" 
        f"Target Code: {item.target_vocabulary_id} | {item.target_concept_id}\n"
        f"Target Name: {item.target_concept_name}\n"
        f"Domain: {item.target_domain_id}\n"
        f"({item.method.value}, confidence: {item.confidence:.2f})\n\n"
    )

Concept Mapping Results:
--------------------------------------------------------------------------------
Source Code = I10
Source Name = Essential (primary) hypertension
Target Code: SNOMED | 320128
Target Name: Essential hypertension
Domain: Condition
(auto, confidence: 8.70)


Source Code = E11.9
Source Name = Type 2 diabetes mellitus without complications
Target Code: SNOMED | 44787902
Target Name: Lipoatrophic diabetes mellitus without complication
Domain: Condition
(auto, confidence: 10.36)


Source Code = R51
Source Name = Headache
Target Code: LOINC | 45883730
Target Name: Headache
Domain: Meas Value
(auto, confidence: 5.01)


Source Code = N18.3
Source Name = Chronic kidney disease stage 3
Target Code: LOINC | 36210709
Target Name: Chronic kidney disease, stage 5
Domain: Meas Value
(auto, confidence: 9.97)


Source Code = F32.9
Source Name = Major depressive disorder single episode unspecified
Target Code: SNOMED | 4282096
Target Name: Major depression, single episode
Domain: 

## Step 10: Deep Dive -- RRF and Cross-Encoder Scores

Each candidate in the mapping results carries multiple score components:

- **`rrf_score`**: The fused rank score from BM25s + FAISS. Higher values mean the candidate
  was ranked highly by both retrieval backends. Computed as `sum(1 / (k + rank_i))`.

- **`cross_encoder_score`**: The raw logit output from the cross-encoder model. This is a
  real-valued number (not bounded to [0,1]) that indicates how strongly the model believes
  the (query, candidate) pair is a match.

- **`score`**: The final blended score used for ranking and confidence routing.
  Computed as `0.6 * sigmoid(cross_encoder_logit) + 0.4 * rrf_score`.

The cross-encoder is the key differentiator. While BM25s and FAISS score candidates
independently (comparing embeddings or term frequencies), the cross-encoder jointly
processes the query and candidate through a single transformer, enabling deep
token-level attention between the two texts.

In [13]:
item.candidates

[ConceptCandidate(concept_id=257011, concept_name='Acute upper respiratory infection', vocabulary_id='SNOMED', domain_id='Condition', concept_class_id='Disorder', standard_concept='S', score=9.532768249511719, rrf_score=9.532768249511719, cross_encoder_score=None),
 ConceptCandidate(concept_id=4183609, concept_name='Influenzal acute upper respiratory infection', vocabulary_id='SNOMED', domain_id='Condition', concept_class_id='Disorder', standard_concept='S', score=8.671640396118164, rrf_score=8.671640396118164, cross_encoder_score=None),
 ConceptCandidate(concept_id=4112341, concept_name='Acute respiratory infections', vocabulary_id='SNOMED', domain_id='Condition', concept_class_id='Disorder', standard_concept='S', score=8.079873085021973, rrf_score=8.079873085021973, cross_encoder_score=None),
 ConceptCandidate(concept_id=4181583, concept_name='Upper respiratory infection', vocabulary_id='SNOMED', domain_id='Condition', concept_class_id='Disorder', standard_concept='S', score=7.873785

In [14]:
for item in concept_map.items[:3]:
    print(f"\n{item.source_code}: {item.target_concept_name}")
    print(f"  Method: {item.method.value}, Confidence: {item.confidence:.4f}")
    if item.candidates:
        print(f"  Top candidates:")
        for c in item.candidates[:3]:
            print(f"    - {getattr(c, 'concept_name', 'N/A')} (score: {getattr(c, 'score', 0):.4f})")


I10: Essential hypertension
  Method: auto, Confidence: 8.6990
  Top candidates:
    - Essential hypertension (score: 8.6990)
    - Systolic essential hypertension (score: 7.7399)
    - Malignant essential hypertension (score: 7.7399)

E11.9: Lipoatrophic diabetes mellitus without complication
  Method: auto, Confidence: 10.3594
  Top candidates:
    - Lipoatrophic diabetes mellitus without complication (score: 10.3594)
    - Diabetes mellitus type 1 without retinopathy (score: 9.6027)
    - Diabetes mellitus type 2 without retinopathy (score: 9.6027)

R51: Headache
  Method: auto, Confidence: 5.0097
  Top candidates:
    - Headache (score: 5.0097)
    - Headaches (score: 5.0097)
    - Headache (score: 5.0097)


In [15]:
# Detailed score breakdown for each candidate (if available)
print("Detailed Score Breakdown (first 3 items):")
print("=" * 90)

for item in concept_map.items[:3]:
    print(f"\nSource: {item.source_code}")
    print(f"  Best match: {item.target_concept_name} (confidence={item.confidence:.4f})")
    if item.candidates:
        print(f"  {'Candidate':<40} {'RRF':>8} {'CE':>8} {'Final':>8}")
        print(f"  {'-'*40} {'-'*8} {'-'*8} {'-'*8}")
        for c in item.candidates[:5]:
            name = c.concept_name[:38]
            rrf = c.rrf_score if c.rrf_score is not None else 0.0
            ce = c.cross_encoder_score if c.cross_encoder_score is not None else 0.0
            final = c.score
            print(f"  {name:<40} {rrf:>8.4f} {ce:>8.4f} {final:>8.4f}")

Detailed Score Breakdown (first 3 items):

Source: I10
  Best match: Essential hypertension (confidence=8.6990)
  Candidate                                     RRF       CE    Final
  ---------------------------------------- -------- -------- --------
  Essential hypertension                     8.6990   0.0000   8.6990
  Systolic essential hypertension            7.7399   0.0000   7.7399
  Malignant essential hypertension           7.7399   0.0000   7.7399
  Benign essential hypertension              7.7399   0.0000   7.7399
  Labile essential hypertension              7.7399   0.0000   7.7399

Source: E11.9
  Best match: Lipoatrophic diabetes mellitus without complication (confidence=10.3594)
  Candidate                                     RRF       CE    Final
  ---------------------------------------- -------- -------- --------
  Lipoatrophic diabetes mellitus without    10.3594   0.0000  10.3594
  Diabetes mellitus type 1 without retin     9.6027   0.0000   9.6027
  Diabetes melli

## Step 11: Compare with Single-Backend Results

To understand the value of hybrid search + reranking, let's compare with a BM25s-only
configuration on the same diagnosis codes.

The cross-encoder reranker is particularly valuable when:

- The correct candidate is retrieved by one backend but not the other
- The correct candidate is in the top-10 but not at position 1
- Two candidates have similar retrieval scores but different semantic relevance

For example, a code like "R51" (Headache) might retrieve both "Headache" (correct)
and "Headache disorder" as top candidates from BM25s. The cross-encoder can distinguish
between these by doing a deeper semantic comparison, potentially promoting the correct
candidate from position 3 to position 1.

In [16]:
# Compare: same codes with BM25s-only (no reranker)
bm25s_config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="bm25s",
        bm25s_corpus_path=str(BM25S_CORPUS),
    ),
    reranker=RerankerConfig(provider="none"),
)

bm25s_project = portiere.init(
    name="BM25s Comparison",
    engine=PolarsEngine(),
    target_model="omop_cdm_v5.4",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    config=bm25s_config,
)

bm25s_diagnoses = bm25s_project.add_source(
    "example_assets/01.3_hybrid_search_reranking/diagnoses.csv"
)

bm25s_concept_map = bm25s_project.map_concepts(source=bm25s_diagnoses)

print("BM25s-Only Results:")
print(f"Summary: {bm25s_concept_map.summary()}")
print()

print("Hybrid + Reranker Results:")
print(f"Summary: {concept_map.summary()}")

2026-04-16 00:31:20 [info     ] PolarsEngine initialized      


2026-04-16 00:31:20 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


2026-04-16 00:31:20 [info     ] project.source_added           project='BM25s Comparison' source=diagnoses


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:31:20 [info     ] gx_profiler.profiled           columns=6 rows=20 source=diagnoses


...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-16 00:31:30 [info     ] Stage 3 complete               auto_rate=100.0% total_codes=8


2026-04-16 00:31:30 [info     ] local_storage.concept_mapping_saved items_count=8 project='BM25s Comparison'


2026-04-16 00:31:30 [info     ] project.concepts_mapped        auto_rate=100.0 project='BM25s Comparison' total=8


BM25s-Only Results:
Summary: {'total': 8, 'auto_mapped': 8, 'needs_review': 0, 'manual_required': 0, 'auto_rate': 100.0, 'coverage': 100.0}

Hybrid + Reranker Results:
Summary: {'total': 8, 'auto_mapped': 8, 'needs_review': 0, 'manual_required': 0, 'auto_rate': 100.0, 'coverage': 100.0}


In [17]:
# Side-by-side comparison of individual mappings
print("Side-by-Side Comparison:")
print("=" * 90)
print(f"{'Code':<10} {'BM25s Match':<30} {'Distance':>6}  {'Hybrid+CE Match':<30} {'Conf':>6}")
print("-" * 90)

bm25s_items = {item.source_code: item for item in bm25s_concept_map.items}
hybrid_items = {item.source_code: item for item in concept_map.items}

all_codes = sorted(set(list(bm25s_items.keys()) + list(hybrid_items.keys())))

for code in all_codes:
    bm25s_item = bm25s_items.get(code)
    hybrid_item = hybrid_items.get(code)

    bm25s_name = (bm25s_item.target_concept_name or "UNMAPPED")[:28] if bm25s_item else "N/A"
    bm25s_conf = bm25s_item.confidence if bm25s_item else 0.0
    hybrid_name = (hybrid_item.target_concept_name or "UNMAPPED")[:28] if hybrid_item else "N/A"
    hybrid_conf = hybrid_item.confidence if hybrid_item else 0.0

    marker = " <<" if hybrid_conf > bm25s_conf else ""
    print(f"  {code:<10} {bm25s_name:<30} {bm25s_conf:>5.2f}  {hybrid_name:<30} {hybrid_conf:>5.2f}{marker}")

Side-by-Side Comparison:
Code       BM25s Match                    Distance  Hybrid+CE Match                  Conf
------------------------------------------------------------------------------------------
  E11.9      Lipoatrophic diabetes mellit   10.36  Lipoatrophic diabetes mellit   10.36
  F32.9      Major depression, single epi   12.28  Major depression, single epi   12.28
  I10        Essential hypertension          8.70  Essential hypertension          8.70
  J06.9      Acute upper respiratory infe    9.53  Acute upper respiratory infe    9.53
  J44.1      Acute exacerbation of chroni   13.55  Acute exacerbation of chroni   13.55
  N18.3      Chronic kidney disease stage    9.97  Chronic kidney disease, stag    9.97
  R51        Headache                        5.01  Headache                        5.01
  Z87.891    Nicotine dependence             8.21  Nicotine dependence             8.21


## Step 12: Approve and Finalize Concept Mappings

After reviewing the hybrid + reranker results, approve all mappings and finalize.
In a production workflow, you would selectively review items below the auto-accept
threshold before approving.

In [18]:
# Check items needing review
review_concepts = concept_map.needs_review()
print(f"Concept mappings needing review: {len(review_concepts)}")
for item in review_concepts:
    print(f"  {item.source_code}: {item.target_concept_name} (confidence: {item.confidence:.4f})")

# Approve all and finalize
concept_map.approve_all()
concept_map.finalize()

print(f"\nConcept mapping finalized.")
print(f"Final summary: {concept_map.summary()}")

Concept mappings needing review: 0

Concept mapping finalized.
Final summary: {'total': 8, 'auto_mapped': 8, 'needs_review': 0, 'manual_required': 0, 'auto_rate': 100.0, 'coverage': 100.0}


## Step 13: Run ETL and Validate

Generate the transformed output files and run validation checks to ensure OMOP CDM
conformance and data quality.

In [19]:
etl_result = project.run_etl(
    patients_source,
    output_dir="./output_hybrid",
    schema_mapping=schema_map,
    concept_mapping=concept_map,
)

print(f"ETL complete. Output: {etl_result}")

2026-04-16 00:31:30 [info     ] Reading source                 format=csv path=example_assets/01.3_hybrid_search_reranking/patients.csv


2026-04-16 00:31:30 [info     ] Processing table               columns=6 table=person


2026-04-16 00:31:30 [info     ] Processing table               columns=4 table=location


2026-04-16 00:31:30 [info     ] ETL complete                   duration=0.004033 success=True


2026-04-16 00:31:30 [info     ] project.etl_complete           output_dir=./output_hybrid project='Hybrid Search Demo'


ETL complete. Output: success=True started_at=datetime.datetime(2026, 4, 15, 17, 31, 30, 106216, tzinfo=datetime.timezone.utc) completed_at=datetime.datetime(2026, 4, 15, 17, 31, 30, 110249, tzinfo=datetime.timezone.utc) duration_seconds=0.004033 source_path='example_assets/01.3_hybrid_search_reranking/patients.csv' source_rows_read=15 output_path='./output_hybrid' tables=[TableResult(table_name='person', rows_written=15, columns=['person_id', 'person_source_value', 'birth_datetime', 'gender_concept_id', 'race_concept_id', 'ethnicity_concept_id'], output_path='./output_hybrid/person.csv', concept_columns_mapped=[], unmapped_concept_count=0), TableResult(table_name='location', rows_written=15, columns=['address_1', 'city', 'state', 'zip'], output_path='./output_hybrid/location.csv', concept_columns_mapped=[], unmapped_concept_count=0)] total_rows_written=30 schema_mappings_applied=10 concept_mappings_applied=0 unmapped_columns=['last_name'] engine_name='polars' errors=[] warnings=[]


In [20]:
validation = project.validate(etl_result=etl_result)

print(f"Validation result: {validation}")

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:31:30 [info     ] gx_validator.validated         completeness=0.44 conformance=1.00 passed=False plausibility=0.44 table=location


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

2026-04-16 00:31:30 [info     ] gx_validator.validated         completeness=0.67 conformance=1.00 passed=False plausibility=0.70 table=person


2026-04-16 00:31:30 [info     ] project.validated              all_passed=False project='Hybrid Search Demo' tables=2


Validation result: {'tables': [{'table_name': 'location', 'passed': False, 'completeness_score': 0.4444444444444444, 'conformance_score': 1.0, 'plausibility_score': 0.4444444444444444, 'gx_result': {'success': False, 'results': [{'success': False, 'expectation_config': {'type': 'expect_column_to_exist', 'kwargs': {'batch_id': 'validate_location-location', 'column': 'location_id'}, 'meta': {}, 'id': '649fc1c8-b30d-458a-8a1b-596d42a6e8c8', 'severity': 'critical'}, 'result': {}, 'meta': {}, 'exception_info': {'raised_exception': False, 'exception_traceback': None, 'exception_message': None}}, {'success': True, 'expectation_config': {'type': 'expect_column_to_exist', 'kwargs': {'batch_id': 'validate_location-location', 'column': 'address_1'}, 'meta': {}, 'id': '7420853f-1c46-47d4-ab0d-f2345760a5af', 'severity': 'critical'}, 'result': {}, 'meta': {}, 'exception_info': {'raised_exception': False, 'exception_traceback': None, 'exception_message': None}}, {'success': False, 'expectation_config

In [21]:
validation

{'tables': [{'table_name': 'location',
   'passed': False,
   'completeness_score': 0.4444444444444444,
   'conformance_score': 1.0,
   'plausibility_score': 0.4444444444444444,
   'gx_result': {'success': False,
    'results': [{'success': False,
      'expectation_config': {'type': 'expect_column_to_exist',
       'kwargs': {'batch_id': 'validate_location-location',
        'column': 'location_id'},
       'meta': {},
       'id': '649fc1c8-b30d-458a-8a1b-596d42a6e8c8',
       'severity': 'critical'},
      'result': {},
      'meta': {},
      'exception_info': {'raised_exception': False,
       'exception_traceback': None,
       'exception_message': None}},
     {'success': True,
      'expectation_config': {'type': 'expect_column_to_exist',
       'kwargs': {'batch_id': 'validate_location-location',
        'column': 'address_1'},
       'meta': {},
       'id': '7420853f-1c46-47d4-ab0d-f2345760a5af',
       'severity': 'critical'},
      'result': {},
      'meta': {},
      'ex

## Step 14: Summary

In this notebook we demonstrated the **highest-accuracy configuration** for OMOP CDM
concept mapping using Portiere's hybrid search with cross-encoder reranking.

### Key Takeaways

- **Hybrid search** combines the strengths of lexical AND semantic search. BM25s catches
  exact term matches, while FAISS captures synonyms, abbreviations, and paraphrases.

- **RRF fusion** ensures concepts found by either retrieval method are considered. A concept
  ranked #1 by BM25s and #5 by FAISS still receives a high fused score.

- **Cross-encoder reranking** provides the highest accuracy by jointly scoring (query, candidate)
  pairs through a single transformer pass. This enables fine-grained token-level comparison
  that bi-encoders and lexical search cannot achieve.

- **Blending formula**: `final = 0.6 * sigmoid(CE) + 0.4 * rrf_score` preserves retrieval
  diversity (40% weight) while letting the cross-encoder make fine-grained corrections
  (60% weight).

- **Best choice when accuracy matters more than speed.** The cross-encoder adds latency
  (~50ms per query vs ~10ms for hybrid-only) but substantially improves precision.

- **Requirements**: `sentence-transformers` + `faiss-cpu` (or `faiss-gpu` for GPU acceleration).

### New Vector Store Options

The `hybrid_backends` parameter now lets you combine **any** dense backend with a sparse
backend. Beyond the classic BM25s + FAISS shown here, you can use:

- `["bm25s", "chromadb"]` -- embedded vector store, no server needed
- `["bm25s", "qdrant"]` -- high-performance search with filtering
- `["bm25s", "milvus"]` -- Milvus Lite for local, or distributed Milvus for scale
- `["elasticsearch", "pgvector"]` -- all-PostgreSQL stack
- `["bm25s", "mongodb"]` -- MongoDB Atlas Vector Search

See `04_knowledge_backends.ipynb` for full configuration details on all 9 backends.

### Decision Matrix

| Use Case | Recommended Backend |
|----------|-------------------|
| Quick prototyping | BM25s |
| Clinical term matching | FAISS |
| Production accuracy | Hybrid + Reranker |
| Speed-critical batch | BM25s or FAISS |

## Next Steps

- **LLM-verified mapping**: See `01.4_llm_verified_mapping.ipynb` to add LLM-based verification
  on top of hybrid search for even higher confidence on ambiguous mappings.
- **Knowledge backends**: See `04_knowledge_backends.ipynb` for a detailed comparison of all
  retrieval backends and their configuration options.
- **Concept mapping deep dive**: See `07_concept_mapping_deep_dive.ipynb` for advanced
  concept mapping workflows including review, override, and export.